# 🎭 Pun Translator (Scalable: Lexique3 + ConceptNet + WordNet)

**Verbose, reproducible pun translation with a default joke-first French rewrite layer.**

This notebook now defaults to **landing the joke in French** instead of preserving literal wording when a pun is detected.


In [11]:
!pip install -q deep-translator requests pandas nltk
print('✅ Installed dependencies')

✅ Installed dependencies


In [12]:
'''
Summary: imports + core classes + translator engine

Load the libraries, ensures WordNet is available, and defines the main translation components:
an English→French translator with caching, a Lexique-based French homophone index, a
ConceptNet-based semantic similarity scorer, and a verbose “polygon” solver that tries
progressively smarter ways to preserve a pun in French.
'''

# Imports + WordNet bootstrap / dependencies

from deep_translator import GoogleTranslator
from typing import List, Optional, Tuple, Dict
from dataclasses import dataclass
import time
import re
import os
import json
import difflib

import pandas as pd
import requests
import nltk
from nltk.corpus import wordnet as wn

# --- NLTK data (WordNet) ---
# Free and only needs to be downloaded once per runtime.
try:
    _ = wn.synsets("bank")
except LookupError:
    nltk.download("wordnet")
    nltk.download("omw-1.4")
    _ = wn.synsets("bank")

In [13]:
# Data models (dataclasses) - structured return objects

@dataclass
class TranslationCandidate:
    pun_word: str
    polygon_level: int
    path: List[str]
    explanation: str
    confidence: float

@dataclass
class FallbackTranslation:
    strategy: str
    translation: str
    explanation: str

In [14]:
# Translation wrapper - cached EN→FR translation.

class RealBilingualDict:
    """Free translation via deep-translator (Google Translate under the hood)."""
    def __init__(self, source_lang='en', target_lang='fr'):
        self.translator = GoogleTranslator(source=source_lang, target=target_lang)
        self.cache: Dict[str, List[str]] = {}
        self.sentence_cache: Dict[str, str] = {}

    def translate(self, word: str) -> List[str]:
        word_lower = word.lower().strip()
        if not word_lower:
            return [word]
        if word_lower in self.cache:
            return self.cache[word_lower]
        try:
            translation = self.translator.translate(word_lower)
            self.cache[word_lower] = [translation]
            time.sleep(0.05)  # be polite to the service
            return [translation]
        except Exception:
            return [word]

    def translate_text(self, text: str) -> str:
        text = (text or "").strip()
        if not text:
            return text
        if text in self.sentence_cache:
            return self.sentence_cache[text]
        try:
            translation = self.translator.translate(text)
            translation = (translation or text).strip()
            self.sentence_cache[text] = translation
            time.sleep(0.05)  # be polite to the service
            return translation
        except Exception:
            return text


In [15]:
# Lexique homophone index - builds the offline phonetic lookup.

class LexiquePhoneticIndex:
    """
    Offline homophone lookup for French using Lexique 3.

    Provide the path to a Lexique 3 file (TSV/CSV). We build:
      phon_form -> [orthographic forms]

    Lexique columns vary slightly by file; we auto-detect likely columns.
    """
    def __init__(self, lexique_path: str, encoding: str = "utf-8"):
        if not os.path.exists(lexique_path):
            raise FileNotFoundError(
                f"Lexique file not found: {lexique_path}\n"
                "Download Lexique 3 from lexique.org and update the path."
            )

        df = self._load_lexique_dataframe(lexique_path, encoding=encoding)

        cols = {str(c).lower(): c for c in df.columns}
        ortho_col = cols.get("ortho") or cols.get("orth") or cols.get("word") or cols.get("lemme") or list(df.columns)[0]
        phon_col = cols.get("phon") or cols.get("phonology") or cols.get("phon_ortho") or cols.get("ipa") or cols.get("phono")
        if phon_col is None:
            raise ValueError(
                "Could not find a phonetic column in the Lexique file.\n"
                "Expected a column like 'phon' (common in Lexique 3).\n"
                f"Columns present: {list(df.columns)[:50]}"
            )

        self.ortho_col = ortho_col
        self.phon_col = phon_col

        self.phon_to_words: Dict[str, List[str]] = {}
        self.word_to_phon: Dict[str, str] = {}

        for _, row in df[[ortho_col, phon_col]].dropna().iterrows():
            w = str(row[ortho_col]).strip().lower()
            p = str(row[phon_col]).strip()
            if not w or not p:
                continue
            self.word_to_phon[w] = p
            self.phon_to_words.setdefault(p, []).append(w)

        for p, words in self.phon_to_words.items():
            seen = set()
            deduped = []
            for w in words:
                if w not in seen:
                    deduped.append(w)
                    seen.add(w)
            self.phon_to_words[p] = deduped

    @staticmethod
    def _load_lexique_dataframe(lexique_path: str, encoding: str = "utf-8") -> pd.DataFrame:
        """
        Load Lexique robustly.

        Some Lexique downloads are TSV, some are CSV, and some contain malformed
        lines that make pandas' default C parser choke. We therefore:
          1) try common separators with the Python engine,
          2) fall back to automatic separator sniffing,
          3) skip malformed lines instead of crashing.
        """
        attempts = [
            {"sep": "\t", "engine": "python"},
            {"sep": ";",  "engine": "python"},
            {"sep": ",",  "engine": "python"},
            {"sep": None, "engine": "python"},  # let pandas sniff
        ]

        last_error = None
        for kwargs in attempts:
            try:
                df = pd.read_csv(
                    lexique_path,
                    encoding=encoding,
                    on_bad_lines="skip",
                    **kwargs,
                )
                if df is not None and len(df.columns) >= 2 and len(df) > 0:
                    return df
            except Exception as e:
                last_error = e

        raise ValueError(
            "Could not parse the Lexique file with the available fallbacks. "
            f"Last parser error: {last_error}"
        )

    def find_homophones(self, french_word: str, limit: int = 30) -> List[str]:
        w = french_word.lower().strip()
        p = self.word_to_phon.get(w)
        if not p:
            return []
        cands = [x for x in self.phon_to_words.get(p, []) if x != w]
        return cands[:limit]


In [16]:
class WordNetSemanticFR:
    """Offline semantic similarity — replaces ConceptNet entirely."""

    def __init__(self):
        self._cache: Dict[Tuple[str,str], float] = {}

    def _synsets_for(self, word: str) -> list:
        ss = wn.synsets(word, lang='fra')
        if ss:
            return ss
        return wn.synsets(word, lang='eng')

    def semantic_similarity(self, word1: str, word2: str) -> float:
        w1, w2 = word1.lower().strip(), word2.lower().strip()
        if not w1 or not w2:
            return 0.0
        if w1 == w2:
            return 1.0
        key = (min(w1,w2), max(w1,w2))
        if key in self._cache:
            return self._cache[key]
        ss1 = self._synsets_for(w1)
        ss2 = self._synsets_for(w2)
        if not ss1 or not ss2:
            score = difflib.SequenceMatcher(None, w1, w2).ratio() * 0.3
            self._cache[key] = score
            return score
        best = 0.0
        for s1 in ss1[:5]:
            for s2 in ss2[:5]:
                try:
                    sim = s1.wup_similarity(s2)
                    if sim and sim > best:
                        best = sim
                except:
                    pass
        self._cache[key] = best
        return best

    def _related_map(self, word: str) -> Dict[str, float]:
        w = word.lower().strip()
        related: Dict[str, float] = {}
        for ss in self._synsets_for(w)[:6]:
            for lem in ss.lemmas():
                name = lem.name().replace('_', ' ').lower()
                if name != w:
                    related[name] = max(related.get(name, 0.0), 8.0)
            for hyper in ss.hypernyms():
                for lem in hyper.lemmas():
                    name = lem.name().replace('_', ' ').lower()
                    related[name] = max(related.get(name, 0.0), 4.0)
            for hypo in ss.hyponyms():
                for lem in hypo.lemmas():
                    name = lem.name().replace('_', ' ').lower()
                    related[name] = max(related.get(name, 0.0), 3.5)
            for sim_ss in ss.also_sees() + ss.similar_tos():
                for lem in sim_ss.lemmas():
                    name = lem.name().replace('_', ' ').lower()
                    related[name] = max(related.get(name, 0.0), 3.0)
        return related

print("✅ WordNetSemanticFR loaded")

✅ WordNetSemanticFR loaded


In [17]:
# ConceptNet semantics - does semantic relatedness via a free API + disk cache.

class ConceptNetSemantic:
    """
    Free semantic relatedness scoring using ConceptNet's public API.

    We approximate similarity by:
    - getting a weighted related-term list for word1,
    - seeing whether word2 appears in that list (and vice versa),
    - using a symmetric score.
    """
    def __init__(self, lang: str = "fr", cache_path: str = "conceptnet_cache.json"):
        self.lang = lang
        self.cache_path = cache_path
        self._cache: Dict[str, Dict[str, float]] = {}
        self._load_cache()

    def _load_cache(self):
        if os.path.exists(self.cache_path):
            try:
                with open(self.cache_path, "r", encoding="utf-8") as f:
                    self._cache = json.load(f)
            except Exception:
                self._cache = {}

    def _save_cache(self):
        try:
            with open(self.cache_path, "w", encoding="utf-8") as f:
                json.dump(self._cache, f, ensure_ascii=False, indent=2)
        except Exception:
            pass

    def _related_map(self, word: str, limit: int = 50) -> Dict[str, float]:
        w = word.lower().strip()
        if not w:
            return {}
        if w in self._cache:
            return self._cache[w]

        concept = f"/c/{self.lang}/{w}"
        url = f"https://api.conceptnet.io/related{concept}?filter=/c/{self.lang}&limit={limit}"

        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            data = r.json()
            rels = {}
            for item in data.get("related", []):
                cid = item.get("@id", "")
                m = re.match(rf"^/c/{self.lang}/(.+)$", cid)
                if not m:
                    continue
                term = m.group(1).replace("_", " ").lower()
                rels[term] = float(item.get("weight", 0.0))
            self._cache[w] = rels
            self._save_cache()
            time.sleep(0.05)
            return rels
        except Exception:
            self._cache[w] = {}
            return {}

    def semantic_similarity(self, word1: str, word2: str) -> float:
        w1 = word1.lower().strip()
        w2 = word2.lower().strip()
        if not w1 or not w2:
            return 0.0
        if w1 == w2:
            return 1.0

        m1 = self._related_map(w1)
        m2 = self._related_map(w2)

        s = max(m1.get(w2, 0.0), m2.get(w1, 0.0))
        return min(1.0, s / 10.0)  # normalize

In [18]:
# Polygon solver - composes the three engines into square/pentagon/hexagon attempts.

class VerboseLowPolygonalTranslator:
    """VERBOSE pun translator with scalable phonetics + semantics."""

    def __init__(self, lexique_path: str):
        self.bilingual_dict = RealBilingualDict('en', 'fr')
        self.phonetic_dict = LexiquePhoneticIndex(lexique_path)
        self.semantic_dict = WordNetSemanticFR()
        self.MIN_SEMANTIC_SIM = 0.05  # ConceptNet-normalized scale

    def _normalize_fr_token(self, text: str) -> str:
        return re.sub(r"[^\w' -]", "", text.lower().strip())

    def _get_french_synonyms(self, french_word: str, min_weight: float = 2.5, limit: int = 12) -> List[str]:
        base = self._normalize_fr_token(french_word)
        related = self.semantic_dict._related_map(base)
        ranked = sorted(related.items(), key=lambda x: x[1], reverse=True)

        # Load a basic French word set to filter out English results
        fr_words = set(self.phonetic_dict.word_to_phon.keys())  # everything in Lexique is French

        out = []
        seen = {base}
        for w, score in ranked:
            w = self._normalize_fr_token(w)
            if not w or w in seen:
                continue
            if " " in w:
                continue
            if score < min_weight:
                continue
            if w not in fr_words:          # ← only keep words that exist in Lexique
                continue
            out.append(w)
            seen.add(w)
            if len(out) >= limit:
                break
        return out

    # ── Low's algorithm: relation-type bonuses ──────────────────────────────
    RELATION_BONUS = {
        "homophone": 0.90,
        "synonym":   0.55,
        "direct":    0.35,
    }

    # Per Low's polygon logic, pattern type shifts how much we weight
    # balance vs. strength vs. relation type.
    # - Word pivot:           single token carries all ambiguity, balance matters most
    # - Phrase pivot:         distributed across a phrase, balance still key
    # - Structure-based:      pun word is a weaker carrier; lean more on strong_meaning as a floor
    # - Full reinterpretation: pun word embedding least reliable; balance is critical
    PUN_PATTERN_WEIGHTS = {
        "word_pivot":            {"balanced": 0.50, "strong": 0.30, "relation": 0.25},
        "phrase_pivot":          {"balanced": 0.50, "strong": 0.25, "relation": 0.25},
        "structure_based":       {"balanced": 0.40, "strong": 0.35, "relation": 0.25},
        "full_reinterpretation": {"balanced": 0.55, "strong": 0.25, "relation": 0.20},
        "unknown":               {"balanced": 0.50, "strong": 0.30, "relation": 0.25},
    }

    @staticmethod
    def _compute_low_score(
        sim1: float,
        sim2: float,
        pun_pattern: str,
        relation_type: str,
        candidate: str,
        base_fr: str,
    ) -> float:
        """
        Composite Low score.

        Replaces the original fixed-weight formula with pattern-aware weights
        so that the metric reflects what Low actually optimises for in each
        pun structure type. The headline output is ``low_score``; raw diffs are
        preserved for downstream compatibility.
        """
        weights = VerboseLowPolygonalTranslator.PUN_PATTERN_WEIGHTS.get(
            pun_pattern,
            VerboseLowPolygonalTranslator.PUN_PATTERN_WEIGHTS["unknown"],
        )

        balanced_meaning     = min(sim1, sim2)
        strong_meaning       = max(sim1, sim2)
        relation_strength    = VerboseLowPolygonalTranslator.RELATION_BONUS.get(relation_type, 0.35)
        same_as_base_penalty = 0.10 if candidate == base_fr else 0.0

        return (
            weights["balanced"]  * balanced_meaning
          + weights["strong"]    * strong_meaning
          + weights["relation"]  * relation_strength
          - same_as_base_penalty
        )

    def _score_low_candidate(
        self,
        candidate: str,
        base_fr: str,
        relation_type: str,
        relation_strength: float,
        t1s: List[str],
        t2s: List[str],
        pun_pattern: str = "unknown",
    ):
        """
        Low-style selection (updated):
        - start from the EN pun word,
        - move to French synonym / homonym candidates,
        - prefer a French pun candidate that covers *both* meanings as well as
          possible, weighted by pun structure type (Low's polygon logic).
        """
        sim1 = max((self.semantic_dict.semantic_similarity(candidate, t) for t in t1s), default=0.0)
        sim2 = max((self.semantic_dict.semantic_similarity(candidate, t) for t in t2s), default=0.0)

        low_score = self._compute_low_score(
            sim1, sim2, pun_pattern, relation_type, candidate, base_fr
        )

        return {
            "candidate":        candidate,
            "base_fr":          base_fr,
            "relation_type":    relation_type,
            "relation_strength": relation_strength,
            "sim1":             sim1,
            "sim2":             sim2,
            "low_score":        low_score,
        }

    def translate_pun_verbose(self, pun_word: str, meaning1: str, meaning2: str, max_polygon: int = 8):
        print(f"\n{'▬'*70}")
        print("🔍 POLYGON TRANSLATION ATTEMPTS")
        print(f"{'▬'*70}")
        print(f"   Pun word: '{pun_word}'")
        print(f"   Meanings: '{meaning1}' ↔ '{meaning2}'")
        print(f"   Will try polygons 4 through {max_polygon}")
        print(f"{'▬'*70}\n")

        for level in range(4, min(max_polygon + 1, 9)):
            polygon_name = ["SQUARE", "PENTAGON", "HEXAGON", "HEPTAGON", "OCTAGON"][level-4]

            print(f"\n🔸 Attempting {polygon_name} ({level}-gon)...")
            print(f"   {'─'*66}")

            if level == 4:
                result = self._attempt_square_verbose(pun_word, meaning1, meaning2)
            elif level == 5:
                result = self._attempt_pentagon_verbose(pun_word, meaning1, meaning2)
            elif level == 6:
                result = self._attempt_hexagon_verbose(meaning1, meaning2)
            elif level == 7:
                result = self._attempt_heptagon_verbose(meaning1, meaning2)
            elif level == 8:
                result = self._attempt_octagon_verbose(meaning1, meaning2)
            else:
                print(f"   Skipping {polygon_name} (not implemented)")
                result = None

            if result:
                print(f"\n   ✅ SUCCESS at {polygon_name}!")
                print(f"   {'─'*66}\n")
                return result, None
            else:
                print(f"   ❌ {polygon_name} failed - no solution found")
                print(f"   {'─'*66}")

        print(f"\n{'='*70}")
        print("⚠️  ALL POLYGONS FAILED (4-8)")
        print(f"{'='*70}")
        print("   Using fallback: LITERAL TRANSLATION")
        print(f"{'='*70}\n")

        t1 = self.bilingual_dict.translate(meaning1)[0]
        fallback = FallbackTranslation(
            strategy="Literal Translation",
            translation=f"{t1}",
            explanation=f"No pun solution found. Translated '{meaning1}' literally to '{t1}'"
        )
        return None, fallback

    def _attempt_square_verbose(self, pun_word: str, m1: str, m2: str):
        """
        Low-style square (updated):
        1) Find the pun word in English
        2) Find synonym or homonym words in French (for the direct translation)
        3) Select the preferred French word using Low's algorithm
        """
        print("   Method: EN pun word → FR synonym/homonym family → Low selection")

        t1s = self.bilingual_dict.translate(m1)
        t2s = self.bilingual_dict.translate(m2)

        print(f"   English pun word: '{pun_word}'")
        print(f"   Meaning A '{m1}' → {t1s}")
        print(f"   Meaning B '{m2}' → {t2s}")

        # Step 1: Direct French renderings of the English pun word
        base_french_forms = [
            self._normalize_fr_token(x)
            for x in self.bilingual_dict.translate(pun_word)
        ]

        base_french_forms = [x for x in dict.fromkeys(base_french_forms) if x]

        print(f"   Direct French forms for '{pun_word}': {base_french_forms}")

        if not base_french_forms:
            print("   No direct French pun candidates found")
            return None

        scored = []
        seen = set()

        for base_fr in base_french_forms:
            # Step 2a: French synonyms / near-synonyms of each base form
            synonym_cands = self._get_french_synonyms(base_fr, min_weight=2.5, limit=12)
            print(f"   Synonyms/related FR words for '{base_fr}': {synonym_cands}")

            # Step 2b: French homophones of each base form
            homophone_cands = self.phonetic_dict.find_homophones(base_fr)
            print(f"   Homophones of '{base_fr}' (Lexique): {homophone_cands[:12]}{'...' if len(homophone_cands) > 12 else ''}")

            # Score the base form itself
            base_key = (base_fr, base_fr, "direct")
            if base_key not in seen:
                seen.add(base_key)
                scored.append(
                    self._score_low_candidate(
                        candidate=base_fr,
                        base_fr=base_fr,
                        relation_type="direct",
                        relation_strength=0.60,
                        t1s=t1s,
                        t2s=t2s
                    )
                )

            # Step 3: Score synonyms via Low's algorithm
            related_map = self.semantic_dict._related_map(base_fr)
            for syn in synonym_cands:
                key = (syn, base_fr, "synonym")
                if key in seen:
                    continue
                seen.add(key)
                strength = min(1.0, related_map.get(syn, 0.0) / 10.0)
                scored.append(
                    self._score_low_candidate(
                        candidate=syn,
                        base_fr=base_fr,
                        relation_type="synonym",
                        relation_strength=strength,
                        t1s=t1s,
                        t2s=t2s
                    )
                )

            # Step 3: Score homophones via Low's algorithm
            for homo in homophone_cands[:20]:
                key = (homo, base_fr, "homophone")
                if key in seen:
                    continue
                seen.add(key)
                scored.append(
                    self._score_low_candidate(
                        candidate=homo,
                        base_fr=base_fr,
                        relation_type="homophone",
                        relation_strength=0.95,
                        t1s=t1s,
                        t2s=t2s
                    )
                )

        if not scored:
            print("   No French synonym/homonym candidates found")
            return None

        scored.sort(key=lambda x: x["low_score"], reverse=True)

        print("   Top Low-style candidates:")
        for row in scored[:8]:
            print(
                f"      {row['candidate']:<18} "
                f"type={row['relation_type']:<9} "
                f"base={row['base_fr']:<12} "
                f"simA={row['sim1']:.2f} "
                f"simB={row['sim2']:.2f} "
                f"Low={row['low_score']:.2f}"
            )

        best = scored[0]

        if max(best["sim1"], best["sim2"]) < self.MIN_SEMANTIC_SIM:
            print("   No candidate passed the semantic threshold")
            return None

        print(
            f"   ✓ Preferred French pun word: '{best['candidate']}' "
            f"(relation={best['relation_type']}, Low={best['low_score']:.2f})"
        )

        return TranslationCandidate(
            pun_word=best["candidate"],
            polygon_level=4,
            path=[pun_word, best["base_fr"], best["relation_type"], best["candidate"], m1, m2],
            explanation=(
                "Square (Low): English pun word → FR synonym/homonym family "
                "→ preferred French pun selection"
            ),
            confidence=min(1.0, best["low_score"])
        )

    def _attempt_pentagon_verbose(self, pun_word: str, m1: str, m2: str):
        """
        Low-style pentagon:
        1) find the pun word in English   -> already supplied as pun_word
        2) find synonym / homonym words in French
        3) select the preferred French pun word
        """
        print("   Method: EN pun word → FR synonym/homonym family → Low selection")

        # Meaning translations are still used to score whether the French candidate
        # can plausibly preserve the two meanings.
        t1s = self.bilingual_dict.translate(m1)
        t2s = self.bilingual_dict.translate(m2)

        print(f"   English pun word: '{pun_word}'")
        print(f"   Meaning A '{m1}' → {t1s}")
        print(f"   Meaning B '{m2}' → {t2s}")

        # Step 1: direct French renderings of the English pun word
        base_french_forms = [
            self._normalize_fr_token(x)
            for x in self.bilingual_dict.translate(pun_word)
        ]
        base_french_forms = [x for x in dict.fromkeys(base_french_forms) if x]

        print(f"   Base French forms for '{pun_word}': {base_french_forms}")

        scored = []
        seen = set()

        for base_fr in base_french_forms:
            # Step 2a: French synonyms / near-synonyms
            synonym_cands = self._get_french_synonyms(base_fr, min_weight=2.5, limit=12)
            print(f"   Synonyms/related FR words for '{base_fr}': {synonym_cands}")

            # Step 2b: French homophones of the base form
            homophone_cands = self.phonetic_dict.find_homophones(base_fr)
            print(f"   Homophones of '{base_fr}' (Lexique): {homophone_cands[:12]}{'...' if len(homophone_cands) > 12 else ''}")

            # Score base itself too, but with weaker relation weight
            base_key = (base_fr, base_fr, "direct")
            if base_key not in seen:
                seen.add(base_key)
                scored.append(
                    self._score_low_candidate(
                        candidate=base_fr,
                        base_fr=base_fr,
                        relation_type="direct",
                        relation_strength=0.35,
                        t1s=t1s,
                        t2s=t2s
                    )
                )

            # Score synonyms
            related_map = self.semantic_dict._related_map(base_fr)
            for syn in synonym_cands:
                key = (syn, base_fr, "synonym")
                if key in seen:
                    continue
                seen.add(key)

                strength = min(1.0, related_map.get(syn, 0.0) / 10.0)
                scored.append(
                    self._score_low_candidate(
                        candidate=syn,
                        base_fr=base_fr,
                        relation_type="synonym",
                        relation_strength=strength,
                        t1s=t1s,
                        t2s=t2s
                    )
                )

                # Also look for homophones of those French synonym candidates
                syn_homophones = self.phonetic_dict.find_homophones(syn)
                if syn_homophones:
                    print(f"   Homophones of synonym '{syn}': {syn_homophones[:8]}{'...' if len(syn_homophones) > 8 else ''}")

                for h in syn_homophones[:10]:
                    hkey = (h, base_fr, "homophone")
                    if hkey in seen:
                        continue
                    seen.add(hkey)
                    scored.append(
                        self._score_low_candidate(
                            candidate=h,
                            base_fr=base_fr,
                            relation_type="homophone",
                            relation_strength=0.95,
                            t1s=t1s,
                            t2s=t2s
                        )
                    )

            # Score direct homophones of the base French form
            for homo in homophone_cands[:20]:
                key = (homo, base_fr, "homophone")
                if key in seen:
                    continue
                seen.add(key)
                scored.append(
                    self._score_low_candidate(
                        candidate=homo,
                        base_fr=base_fr,
                        relation_type="homophone",
                        relation_strength=0.95,
                        t1s=t1s,
                        t2s=t2s
                    )
                )

        if not scored:
            print("   No French synonym/homonym candidates found")
            return None

        scored.sort(key=lambda x: x["low_score"], reverse=True)

        print("   Top Low-style candidates:")
        for row in scored[:8]:
            print(
                f"      {row['candidate']:<18} "
                f"type={row['relation_type']:<9} "
                f"base={row['base_fr']:<12} "
                f"simA={row['sim1']:.2f} "
                f"simB={row['sim2']:.2f} "
                f"Low={row['low_score']:.2f}"
            )

        best = scored[0]

        # Require some real evidence that the candidate preserves meaning
        if max(best["sim1"], best["sim2"]) < self.MIN_SEMANTIC_SIM:
            print("   No candidate passed the semantic threshold")
            return None

        print(
            f"   ✓ Preferred French pun word: '{best['candidate']}' "
            f"(relation={best['relation_type']}, Low={best['low_score']:.2f})"
        )

        return TranslationCandidate(
            pun_word=best["candidate"],
            polygon_level=5,
            path=[pun_word, best["base_fr"], best["relation_type"], best["candidate"], m1, m2],
            explanation=(
                "Pentagon (Low): English pun word → French synonym/homonym family "
                "→ preferred French pun selection"
            ),
            confidence=min(1.0, best["low_score"])
        )

    def _attempt_hexagon_verbose(self, m1: str, m2: str):
        print("   Method: Translate → synonym (WordNet EN) → translate → homophone (Lexique) → semantic check (ConceptNet)")
        # Step 1: translate target meaning (B) once
        t2s = self.bilingual_dict.translate(m2)
        print(f"   Target meaning '{m2}' → {t2s}")

        # Step 2: generate English synonym candidates for m1 (plus m1 itself)
        base = m1.lower().strip()
        syns = set()
        for ss in wn.synsets(base):
            for lem in ss.lemmas():
                name = lem.name().replace("_", " ").lower().strip()
                # keep simple single-token synonyms to reduce drift
                if name.isalpha() and 3 <= len(name) <= 20:
                    syns.add(name)
        syn_list = [base] + sorted(syns - {base})
        syn_list = syn_list[:12]  # keep bounded & fast
        print(f"   Synonym candidates for '{m1}': {syn_list}")

        # Step 3: for each synonym, translate to French, then search homophones and validate semantically
        for syn_en in syn_list:
            fr_syns = self.bilingual_dict.translate(syn_en)
            print(f"   '{syn_en}' → {fr_syns}")

            for fr in fr_syns:
                homophones = self.phonetic_dict.find_homophones(fr)
                if homophones:
                    preview = homophones[:12]
                    print(f"   Homophones of '{fr}' (Lexique): {preview}{'...' if len(homophones) > 12 else ''}")
                else:
                    print(f"   Homophones of '{fr}' (Lexique): []")
                    continue

                for homophone in homophones:
                    for t2 in t2s:
                        sim = self.semantic_dict.semantic_similarity(homophone, t2)
                        print(f"   Similarity('{homophone}', '{t2}'): {sim:.2f}")
                        if sim >= self.MIN_SEMANTIC_SIM:
                            print(f"   ✓ Semantic match! {sim:.2f} ≥ {self.MIN_SEMANTIC_SIM}")
                            return TranslationCandidate(
                                pun_word=homophone, polygon_level=6,
                                path=[m1, syn_en, fr, homophone, t2, m2],
                                explanation="Hexagon: synonym → translation → homophone → semantic match",
                                confidence=sim
                            )

        print("   No synonym/homophone passed semantic threshold")
        return None

    def _attempt_heptagon_verbose(self, m1: str, m2: str):
        """
        Heptagon (7-gon): 3 semantic/phonetic leaps
        Path: m1 → t1 → FR_synonym → homophone → FR_synonym → t2 → m2
        """
        print("   Method: Translate → FR synonym → homophone → FR synonym → check")

        t1s = self.bilingual_dict.translate(m1)
        t2s = self.bilingual_dict.translate(m2)
        print(f"   '{m1}' → {t1s}")
        print(f"   '{m2}' → {t2s}")

        for t1 in t1s[:3]:
            related1 = self.semantic_dict._related_map(t1)
            fr_syns1 = [w for w, score in related1.items() if score > 3.0][:10]

            if fr_syns1:
                print(f"   FR synonyms of '{t1}': {fr_syns1[:5]}...")

            for fr_syn in fr_syns1:
                homos = self.phonetic_dict.find_homophones(fr_syn)

                if homos:
                    print(f"   Homophones of '{fr_syn}': {homos[:5]}...")

                for homo in homos[:10]:
                    related_h = self.semantic_dict._related_map(homo)
                    fr_syns_h = [w for w, score in related_h.items() if score > 3.0][:10]

                    for syn_h in fr_syns_h:
                        for t2 in t2s:
                            sim = self.semantic_dict.semantic_similarity(syn_h, t2)

                            if sim >= self.MIN_SEMANTIC_SIM:
                                print(f"   ✓ 7-gon match! {sim:.2f} ≥ {self.MIN_SEMANTIC_SIM}")
                                return TranslationCandidate(
                                    pun_word=homo,
                                    polygon_level=7,
                                    path=[m1, t1, fr_syn, homo, syn_h, t2, m2],
                                    explanation="Heptagon: 3 leaps (synonym→homophone→synonym)",
                                    confidence=sim * 0.8
                                )

        print("   No 7-gon path found")
        return None


    def _attempt_octagon_verbose(self, m1: str, m2: str):
        """
        Octagon (8-gon): 4 semantic/phonetic leaps
        Path: m1 → t1 → syn1 → homo1 → syn2 → homo2 → t2 → m2
        """
        print("   Method: Translate → syn → homo → syn → homo → check")
        print("   ⚠️  8-gon paths are very creative (potentially tenuous)")

        t1s = self.bilingual_dict.translate(m1)
        t2s = self.bilingual_dict.translate(m2)
        print(f"   '{m1}' → {t1s}")
        print(f"   '{m2}' → {t2s}")

        # Very limited search
        for t1 in t1s[:2]:
            related1 = self.semantic_dict._related_map(t1)
            syns1 = [w for w, score in related1.items() if score > 4.0][:5]

            if syns1:
                print(f"   Synonyms of '{t1}': {syns1[:3]}...")

            for syn1 in syns1:
                homos1 = self.phonetic_dict.find_homophones(syn1)[:5]

                for homo1 in homos1:
                    related2 = self.semantic_dict._related_map(homo1)
                    syns2 = [w for w, score in related2.items() if score > 4.0][:5]

                    for syn2 in syns2:
                        homos2 = self.phonetic_dict.find_homophones(syn2)[:5]

                        for homo2 in homos2:
                            for t2 in t2s:
                                sim = self.semantic_dict.semantic_similarity(homo2, t2)

                                if sim >= self.MIN_SEMANTIC_SIM:
                                    print(f"   ✓ 8-gon match! {sim:.2f} ≥ {self.MIN_SEMANTIC_SIM}")
                                    return TranslationCandidate(
                                        pun_word=homo2,
                                        polygon_level=8,
                                        path=[m1, t1, syn1, homo1, syn2, homo2, t2, m2],
                                        explanation="Octagon: 4 leaps (very creative path)",
                                        confidence=sim * 0.6
                                    )

        print("   No 8-gon path found")
        return None

    def translate_sentence(self, sentence: str, pun_word_original: str, pun_word_french: str) -> str:
        base = self.bilingual_dict.translate_text(sentence)
        if not base:
            return base
        # Best effort: swap in the selected French pun word if the original English surface survives badly.
        pattern = re.compile(rf"{re.escape(pun_word_original)}", flags=re.I)
        if pattern.search(base):
            return pattern.sub(pun_word_french, base)
        return base

print("✅ Scalable translator loaded (Lexique + ConceptNet + WordNet)")

✅ Scalable translator loaded (Lexique + ConceptNet + WordNet)


In [19]:
'''
Summary: get_cosine_similarity — Low-score cosine evaluation

Evaluates how well an English pun and its French candidate each straddle
*both* meanings of the pun (Low\'s core criterion).  The headline output is
``low_score_diff`` (EN minus FR), which replaces the thin ``first/second_
similarity_diff`` signals used in earlier versions.

Weights are driven by:
  • pun_pattern  — structure type from AutoPunDetector
  • pun_type     — homophone / synonym / direct (relation type)
See VerboseLowPolygonalTranslator.PUN_PATTERN_WEIGHTS for the rationale.
'''

import ast
import torch
from sentence_transformers import SentenceTransformer, util


def get_model(model_name: str) -> SentenceTransformer:
    """Cache sentence-transformer models by name."""
    if not hasattr(get_model, "_cache"):
        get_model._cache = {}
    if model_name not in get_model._cache:
        get_model._cache[model_name] = SentenceTransformer(model_name)
    return get_model._cache[model_name]


def get_cosine_similarity(df, model: str, start: int = 0, end: int = -1, similarity_dir: str = "similarities/"):
    """
    Compute Low-score cosine similarity for each row of *df*.

    Required columns
    ----------------
    pun_word, first_meaning, second_meaning   — English pun and its two senses
                                                (first/second_meaning are JSON-
                                                serialised lists of sense words)
    pun_word_fr, first_meaning_fr,
    second_meaning_fr                         — French equivalents
    pun_pattern   (optional)                  — word_pivot / phrase_pivot /
                                                structure_based /
                                                full_reinterpretation / unknown
    pun_type      (optional)                  — homophone / synonym / direct
    base_fr       (optional)                  — the direct FR translation of the
                                                pun word (for the same-as-base
                                                penalty)

    Output columns
    --------------
    first_similarity_en / fr / diff   — raw cosine similarities + diff
    second_similarity_en / fr / diff
    low_score_en / fr / diff          — Low composite score (headline metric)
    """

    def _compute_low_score(sim1, sim2, pun_type, relation_type, pun_word, base):
        """
        Composite Low score (mirrors VerboseLowPolygonalTranslator._compute_low_score).

        Weights are pattern-aware so the metric reflects what Low actually
        optimises for in each pun structure type.
        """
        weights = VerboseLowPolygonalTranslator.PUN_PATTERN_WEIGHTS.get(
            pun_type,
            VerboseLowPolygonalTranslator.PUN_PATTERN_WEIGHTS["unknown"],
        )

        balanced_meaning     = min(sim1, sim2)
        strong_meaning       = max(sim1, sim2)
        relation_strength    = VerboseLowPolygonalTranslator.RELATION_BONUS.get(relation_type, 0.35)
        same_as_base_penalty = 0.10 if pun_word == base else 0.0

        return (
            weights["balanced"]  * balanced_meaning
          + weights["strong"]    * strong_meaning
          + weights["relation"]  * relation_strength
          - same_as_base_penalty
        )

    def _apply(row, st_model):
        # ── English embeddings ──────────────────────────────────────────
        pun_emb_en    = st_model.encode([row["pun_word"]], convert_to_tensor=True)
        m1_emb_en     = torch.mean(
            st_model.encode(ast.literal_eval(row["first_meaning"]), convert_to_tensor=True),
            dim=0, keepdim=True)
        m2_emb_en     = torch.mean(
            st_model.encode(ast.literal_eval(row["second_meaning"]), convert_to_tensor=True),
            dim=0, keepdim=True)

        # ── French embeddings ───────────────────────────────────────────
        pun_emb_fr    = st_model.encode([row["pun_word_fr"]], convert_to_tensor=True)
        m1_emb_fr     = torch.mean(
            st_model.encode(ast.literal_eval(row["first_meaning_fr"]), convert_to_tensor=True),
            dim=0, keepdim=True)
        m2_emb_fr     = torch.mean(
            st_model.encode(ast.literal_eval(row["second_meaning_fr"]), convert_to_tensor=True),
            dim=0, keepdim=True)

        # ── Raw cosine similarities ─────────────────────────────────────
        first_sim_en  = util.cos_sim(pun_emb_en, m1_emb_en).item()
        second_sim_en = util.cos_sim(pun_emb_en, m2_emb_en).item()
        first_sim_fr  = util.cos_sim(pun_emb_fr, m1_emb_fr).item()
        second_sim_fr = util.cos_sim(pun_emb_fr, m2_emb_fr).item()

        first_diff    = first_sim_en  - first_sim_fr
        second_diff   = second_sim_en - second_sim_fr

        # ── Low composite scores ────────────────────────────────────────
        pun_pattern   = row.get("pun_pattern", "unknown")
        relation_type = row.get("pun_type",    "direct")
        base_fr       = row.get("base_fr",     row["pun_word_fr"])

        low_en   = _compute_low_score(
            first_sim_en, second_sim_en,
            pun_pattern, relation_type,
            row["pun_word"], row["pun_word"],   # EN word is its own base
        )
        low_fr   = _compute_low_score(
            first_sim_fr, second_sim_fr,
            pun_pattern, relation_type,
            row["pun_word_fr"], base_fr,
        )
        low_diff = low_en - low_fr

        print(row.name, row["pun_word"], row["pun_word_fr"], pun_pattern, relation_type)
        print("first  en", first_sim_en,  "fr", first_sim_fr,  "diff", first_diff)
        print("second en", second_sim_en, "fr", second_sim_fr, "diff", second_diff)
        print("low    en", low_en, "fr", low_fr, "diff", low_diff)

        return pd.Series({
            "first_similarity_en":    first_sim_en,
            "second_similarity_en":   second_sim_en,
            "first_similarity_fr":    first_sim_fr,
            "second_similarity_fr":   second_sim_fr,
            "first_similarity_diff":  first_diff,
            "second_similarity_diff": second_diff,
            "low_score_en":           low_en,
            "low_score_fr":           low_fr,
            "low_score_diff":         low_diff,
        })

    import os, pandas as _pd

    st_model   = get_model(model)
    chunk_size = 10
    chunks     = [df.iloc[i:i + chunk_size] for i in range(0, len(df), chunk_size)]
    if end == -1:
        end = len(chunks)

    for i in range(start, end):
        current_df = chunks[i].copy()
        current_df[
            ["first_similarity_en", "second_similarity_en",
             "first_similarity_fr", "second_similarity_fr",
             "first_similarity_diff", "second_similarity_diff",
             "low_score_en", "low_score_fr", "low_score_diff"]
        ] = current_df.apply(_apply, axis=1, args=(st_model,))

        out_path = f"{similarity_dir}{model}/{i}.tsv"
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        current_df.to_csv(out_path, sep="\t", index=False)

    return df


print("✅ get_cosine_similarity loaded (Low-score evaluation)")


✅ get_cosine_similarity loaded (Low-score evaluation)


In [20]:

'''
Summary: AutoPunDetector

This cell defines the pun-word detector that tokenizes the sentence, filters stopwords, then
either flags a plausible out-of-vocabulary pun candidate, or falls back to WordNet to pick a
real content word with two far-apart senses.

UPDATED:
- function words like "what" are no longer treated as joke targets
- OOV candidates are scored instead of accepted immediately
- final/payoff-position words get a boost, which helps catch punchlines like "offal"
'''

class AutoPunDetector:
    """Automatically proposes a pun word + two distinct meanings."""

    def __init__(self):
        self.stop = {
            "the","a","an","and","or","but","if","then","else","to","of","in","on","at","for","with",
            "is","are","was","were","be","been","being","it","this","that","these","those",
            "i","you","he","she","we","they","me","him","her","us","them","my","your","his","their",
            "as","by","from","not","no","so","what","which","who","whom","whose","where","when","why","how",
            "just","only","very","really","there","here","than","then"
        }

        lemmas = set()
        for name in wn.all_lemma_names():
            if not name or "_" in name:
                continue
            n = name.lower()
            if not n.isalpha() or len(n) < 3:
                continue
            lemmas.add(n)
        self.wn_vocab = sorted(lemmas)

    def _tokenize(self, sentence: str) -> List[str]:
        return re.findall(r"[A-Za-z']+", sentence.lower())

    def _clean_tokens(self, sentence: str) -> List[str]:
        toks = self._tokenize(sentence)
        return [t for t in toks if t and t not in self.stop]

    def _test_in_context(self, candidate: str, oov_word: str, sentence: str) -> float:
        score = 0.0
        sentence_lower = sentence.lower()
        word_pos = sentence_lower.find(oov_word.lower())
        if word_pos == -1:
            return 0.0

        before_text = sentence_lower[:word_pos].strip()
        before_words = before_text.split() if before_text else []

        if len(before_words) >= 2 and before_words[-1] == 'the' and before_words[-2] in ['did', 'do', 'does']:
            achievement_words = {
                'impossible', 'possible', 'incredible', 'unbelievable',
                'remarkable', 'improbable', 'unthinkable', 'unexpected'
            }
            if candidate in achievement_words:
                score += 5.0
            else:
                score -= 2.0

        if len(before_words) >= 2 and before_words[-1] in ['an', 'a'] and before_words[-2] in ['is', 'was', 'are']:
            noun_endings = candidate.endswith(('a', 'o', 'le', 'er', 'or', 'ist'))
            if noun_endings or candidate in {'pasta', 'noodle', 'impasta', 'impostor'}:
                score += 4.0

        sentence_words = set(sentence_lower.split())
        food_indicators = {'noodle', 'pasta', 'food', 'eat', 'fake', 'dish', 'meal', 'cook'}
        food_words = {'pasta', 'noodle', 'impasta', 'spaghetti'}
        if (sentence_words & food_indicators) and (candidate in food_words):
            score += 4.0

        similarity = difflib.SequenceMatcher(None, oov_word.lower(), candidate).ratio()
        score += similarity * 1.5
        return score

    def _is_oov(self, word: str) -> bool:
        if word in self.stop or len(word) < 4:
            return False
        return len(wn.synsets(word)) == 0

    def _surface_position_bonus(self, word: str, sentence: str) -> float:
        toks = self._clean_tokens(sentence)
        if not toks or word not in toks:
            return 0.0
        idx = max(i for i, t in enumerate(toks) if t == word)
        bonus = 0.0
        if idx >= len(toks) - 2:
            bonus += 4.0
        if re.search(r"\b(but|however|though|yet)\b", sentence.lower()):
            tail = toks[len(toks)//2:]
            if word in tail:
                bonus += 2.5
        if re.search(rf"\bjust\s+{re.escape(word)}\b", sentence.lower()):
            bonus += 2.0
        return bonus

    def _oov_viability_score(self, tok: str, sentence: str) -> float:
        if tok in self.stop or len(tok) < 5:
            return -999.0
        score = 0.0
        score += self._surface_position_bonus(tok, sentence)
        vowel_count = sum(ch in "aeiouy" for ch in tok)
        if vowel_count >= 2:
            score += 0.5
        if tok.endswith(("al", "er", "or", "ing", "ster")):
            score += 0.75
        if "'" in tok:
            score -= 1.5

        synsets = wn.synsets(tok)
        if synsets:
            score += min(4.0, len(synsets) * 0.8)

            noun_or_adj = any(s.pos() in {"n", "a", "s"} for s in synsets)
            if noun_or_adj:
                score += 1.5

            defs = " ".join(s.definition().lower() for s in synsets[:4])
            domain_hits = {
                "meat","animal","organ","viscera","guts","waste","plant","money","river",
                "school","trip","field","processing","factory","gross","awful","body"
            }
            overlap = sum(1 for w in domain_hits if w in defs or w in sentence.lower())
            score += min(3.0, overlap * 0.5)
        else:
            if self._detect_oov_pun_context_aware(tok, sentence, verbose=False):
                score += 3.0

        return score

    def _sense_label(self, synset, surface: str) -> str:
        first_lemma = synset.lemmas()[0].name().replace("_", " ").lower()
        if first_lemma == surface.lower():
            definition = synset.definition()
            stop = {"a", "an", "the", "of", "in", "on", "to", "or", "and", "for",
                    "with", "at", "by", "as", "is", "are", "that", "which"}
            for word in definition.split():
                word_clean = re.sub(r"[^a-z]", "", word.lower())
                if word_clean and word_clean not in stop and len(word_clean) > 2:
                    return word_clean
            return " ".join(definition.split()[:4])
        return first_lemma

    def _best_synset_pair(self, synsets):
        best = None
        best_d = -1
        for i in range(len(synsets)):
            for j in range(i + 1, len(synsets)):
                d = synsets[i].shortest_path_distance(synsets[j])
                if d is None:
                    d = 10
                pos_bonus = 5 if synsets[i].pos() != synsets[j].pos() else 0
                effective_d = d + pos_bonus
                if effective_d > best_d:
                    best_d = effective_d
                    best = (synsets[i], synsets[j], d)
        return best

    def _detect_oov_pun_context_aware(self, tok: str, sentence: str, verbose: bool = True) -> Optional[Tuple[str, str, str, str]]:
        if verbose:
            print(f"   '{tok}' is OOV → testing portmanteau candidates with context...")

        all_candidates = set()
        for i in range(len(tok)):
            for j in range(i + 3, len(tok) + 1):
                substring = tok[i:j]
                if substring in self.wn_vocab:
                    all_candidates.add(substring)

        close = difflib.get_close_matches(tok, self.wn_vocab, n=20, cutoff=0.55)
        all_candidates.update(close)

        if not all_candidates:
            if verbose:
                print(f"   ⚠️  No candidates found")
            return None

        if verbose:
            print(f"   Testing {len(all_candidates)} candidates in sentence context...")

        scored = []
        for cand in all_candidates:
            context_score = self._test_in_context(cand, tok, sentence)
            if context_score > 1.0:
                scored.append((cand, context_score))

        scored.sort(key=lambda x: x[1], reverse=True)

        if verbose and scored:
            print(f"\n   📊 Top candidates by context score:")
            for i, (cand, sc) in enumerate(scored[:5], 1):
                print(f"      {i}. '{cand}' → {sc:.2f}")

        if len(scored) >= 2:
            meaning_a = scored[0][0]
            meaning_b = None
            for cand, sc in scored[1:]:
                if difflib.SequenceMatcher(None, meaning_a, cand).ratio() < 0.5:
                    meaning_b = cand
                    break
            if not meaning_b:
                meaning_b = scored[1][0]

            explanation = f"Portmanteau: {meaning_a} + {meaning_b} (context-aware)"
            if verbose:
                print(f"\n   ✅ Selected: '{meaning_a}' + '{meaning_b}'")
            return (tok, meaning_a, meaning_b, explanation)

        if len(scored) == 1:
            meaning = scored[0][0]
            explanation = f"Portmanteau: {meaning} (single candidate)"
            return (tok, meaning, meaning, explanation)

        return None

    def detect(self, sentence: str) -> Optional[Tuple[str, str, str, str]]:
        tokens = self._clean_tokens(sentence)
        if not tokens:
            return None

        counts: Dict[str, int] = {}
        for t in tokens:
            counts[t] = counts.get(t, 0) + 1

        best = None

        # 1) Score plausible OOV candidates, but do not let weak ones hijack the sentence.
        for tok in tokens:
            if not self._is_oov(tok):
                continue
            viability = self._oov_viability_score(tok, sentence)
            if viability < 4.5:
                continue
            result = self._detect_oov_pun_context_aware(tok, sentence)
            if result:
                _, m1, m2, explain = result
                cand = (viability + 15.0, counts.get(tok, 1), viability, tok, m1, m2, explain)
                if (best is None) or (cand[:3] > best[:3]):
                    best = cand

        # 2) Score real-word polysemy candidates, with a boost for end-of-punchline position.
        for w, cnt in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
            synsets = wn.synsets(w)
            if len(synsets) < 1:
                continue

            if len(synsets) >= 2:
                pair = self._best_synset_pair(synsets[:10])
            else:
                pair = None

            if pair:
                s1, s2, dist = pair
                m1 = self._sense_label(s1, w)
                m2 = self._sense_label(s2, w)
                if m1 == m2:
                    m1 = " ".join(s1.definition().split()[:5])
                    m2 = " ".join(s2.definition().split()[:5])
                score = dist + 15 * max(0, cnt - 1) + min(3.0, 0.25 * len(synsets))
            else:
                # single-sense but punchline-heavy content words can still be joke carriers
                s1 = synsets[0]
                m1 = self._sense_label(s1, w)
                defs = s1.definition().lower()
                if re.search(r"(viscera|organ|entrails|animal|waste|meat)", defs):
                    m2 = "awful"
                    score = 9.5
                else:
                    continue

            score += self._surface_position_bonus(w, sentence)

            explanation = (
                f"Auto-detected '{w}' as a possible pun word (score {score:.2f}).\n"
                f"Meaning A: {m1}\n"
                f"Meaning B: {m2}"
            )

            cand = (score, cnt, score, w, m1, m2, explanation)
            if (best is None) or (cand[:3] > best[:3]):
                best = cand

        if not best:
            return None

        _, _, _, w, m1, m2, explanation = best
        return (w, m1, m2, explanation)

print("✅ AutoPunDetector loaded (function-word guardrails + payoff weighting)")


✅ AutoPunDetector loaded (Context-Aware OOV + WordNet-based)


In [21]:
'''
Summary: translate_pun_complete orchestrator

This cell is the end-to-end pipeline: it prints the input, calls the auto pun detector, runs
the polygon translator on the two meanings if a pun is found, and then defaults to a joke-first
French rewrite that tries to preserve the laugh instead of the literal wording.
'''

def translate_pun_complete(sentence: str, lexique_path: str, show_details: bool = True):
    """Translate a sentence, attempting an automatically detected pun first."""

    print(f"\n{'='*70}")
    print("📝 ENGLISH INPUT")
    print(f"{'='*70}")
    print(f"   {sentence}")
    print(f"{'='*70}")

    detector = AutoPunDetector()
    result = detector.detect(sentence)

    if not result:
        print(f"\n{'='*70}")
        print("⚠️  NO PUN CANDIDATE DETECTED - DOING LITERAL TRANSLATION")
        print(f"{'='*70}")
        translator = RealBilingualDict('en','fr')
        literal = translator.translate_text(sentence)
        print("\nWhole-sentence translation:")
        print(f"   {literal}")
        return literal

    pun_word, meaning1, meaning2, explain = result

    print(f"\n{'='*70}")
    print("🎯 PUN DETECTED (AUTO)")
    print(f"{'='*70}")
    print(f"   Pun word: {pun_word}")
    print(f"   Meaning A: {meaning1}")
    print(f"   Meaning B: {meaning2}")
    print(f"\n   Details:\n{explain}")
    print(f"{'='*70}")

    translator = VerboseLowPolygonalTranslator(lexique_path=lexique_path)
    candidate, fallback = translator.translate_pun_verbose(pun_word, meaning1, meaning2)

    joke_first = build_joke_first_french(sentence, pun_word, meaning1, meaning2, translator, candidate)

    if candidate:
        print(f"\n{'='*70}")
        print("✅ PUN CANDIDATE FOUND")
        print(f"{'='*70}")
        print(f"   French pun word: {candidate.pun_word}")
        print(f"   Confidence: {candidate.confidence:.2f}")
        print(f"   Path: {' → '.join(candidate.path)}")
        print(f"   Explanation: {candidate.explanation}")
        print(f"{'='*70}")

    if joke_first:
        print(f"\n{'='*70}")
        print("😂 DEFAULT JOKE-FIRST FRENCH")
        print(f"{'='*70}")
        print("   Returning a laugh-first French rewrite rather than a literal transfer.")
        print(f"{'='*70}")
        print(f"\n📌 Full sentence: {joke_first}")
        return joke_first

    if candidate:
        french_sentence = translator.translate_sentence(sentence, pun_word, candidate.pun_word)
        print(f"\n📌 Full sentence: {french_sentence}")
        return french_sentence

    print(f"\n{'='*70}")
    print("⚠️  FALLBACK (NO PUN FOUND)")
    print(f"{'='*70}")
    print(f"   Strategy: {fallback.strategy}")
    print(f"   Explanation: {fallback.explanation}")
    print(f"{'='*70}")

    translator2 = RealBilingualDict('en','fr')
    return translator2.translate_text(sentence)


In [ ]:

'''
Summary: default joke-first French rewrite layer

Builds a generic, non-hardcoded French punchline when a pun is detected.
The goal is not literal accuracy; it is to make the translation sound like
it is trying to get a laugh in French by default.

UPDATED:
- translates the setup clause as a full phrase instead of word by word
- applies light French clean-up passes
- uses better noun article selection in the punchline
'''

def _normalize_token_list(words):
    out = []
    seen = set()
    for w in words:
        w = re.sub(r"[^\w' -]", "", (w or "").lower()).strip()
        if not w or len(w) < 2:
            continue
        if w in seen:
            continue
        seen.add(w)
        out.append(w)
    return out

def _split_on_contrast(sentence: str):
    s = sentence.strip()
    parts = re.split(r"\b(but|however|though|yet|except|although|while)\b", s, flags=re.I, maxsplit=1)
    if len(parts) >= 3:
        return parts[0].strip(" ,;—-"), parts[2].strip(" ,;—-")
    return s, ""

def _compact_keywords_fr(text: str, translator_obj, stop_words):
    toks = re.findall(r"[A-Za-z']+", text.lower())
    content = [t for t in toks if t not in stop_words and len(t) > 2]
    translated = [translator_obj.translate(t)[0] for t in content]
    translated = _normalize_token_list(translated)
    return translated

def _pick_tail_noun(pun_word: str, meaning1: str, meaning2: str, sentence: str, translator_obj, stop_words):
    sources = [meaning1, meaning2, pun_word, sentence]
    candidates = []
    for src in sources:
        for tok in re.findall(r"[A-Za-z']+", src.lower()):
            if tok in stop_words or len(tok) < 3:
                continue
            fr = translator_obj.translate(tok)[0]
            fr = re.sub(r"[^\w' -]", "", fr.lower()).strip()
            if fr and fr not in candidates:
                candidates.append(fr)

    preference_patterns = [
        r"d[ée]chets?$", r"abats?$", r"tripes?$", r"ordures?$", r"restes?$",
        r"viande$", r"banque$", r"rive$", r"argent$", r"sale$"
    ]
    for pat in preference_patterns:
        for cand in candidates:
            if re.search(pat, cand):
                return cand
    return candidates[0] if candidates else "truc"

def _classify_vibe(sentence: str, meaning1: str, meaning2: str):
    text = " ".join([sentence.lower(), meaning1.lower(), meaning2.lower()])
    negative = {
        "awful","offal","waste","wasted","garbage","gross","disgust","dirty","blood","dead",
        "corpse","rot","trash","bad","ugly","stink","guts","organ","meat","slaughter","toxic"
    }
    positive = {
        "smart","clever","brilliant","funny","rich","lucky","love","good","great","excellent",
        "talent","skill","charm","beauty","success"
    }
    absurd = {"weird","odd","strange","absurd","nonsense","crazy","wild","twist"}

    neg_score = sum(1 for w in negative if w in text)
    pos_score = sum(1 for w in positive if w in text)
    absurd_score = sum(1 for w in absurd if w in text)

    if neg_score >= max(pos_score, absurd_score) and neg_score > 0:
        return "negative"
    if pos_score >= max(neg_score, absurd_score) and pos_score > 0:
        return "positive"
    if absurd_score > 0:
        return "absurd"
    return "neutral"

def _choose_french_quantifier(noun: str) -> str:
    noun = (noun or "").strip().lower()
    pluralish = noun.endswith("s") or noun in {"déchets", "abats", "tripes", "ordures", "restes"}
    if pluralish:
        return f"que des {noun}"
    vowels = tuple("aeiouyhàâäéèêëîïôöùûü")
    if noun.startswith(vowels):
        return f"que de l'{noun}"
    if noun.endswith(("e", "ion", "ure", "té")):
        return f"que de la {noun}"
    return f"que du {noun}"

def _cleanup_french_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\b[Cc]e qu'ils\b", "Ce qu’ils", text)
    replacements = {
        "Le classe": "La classe",
        "champ voyage": "voyage scolaire",
        "viande traitement usine": "usine de transformation de viande",
        "du déchets": "des déchets",
        "de déchets": "des déchets",
        "que du déchets": "que des déchets",
        "qu'ils": "qu’ils",
        "c'etait": "c’était",
        "c'était": "c’était",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    text = text.replace(" ,", ",").replace(" .", ".").replace(" :", " :")
    text = re.sub(r"\.{3,}", "…", text)
    text = re.sub(r"^([a-zàâçéèêëîïôûùüÿñæœ])", lambda m: m.group(1).upper(), text)
    return text

def build_joke_first_french(sentence: str, pun_word: str, meaning1: str, meaning2: str, polygon_translator, candidate=None):
    bilingual = polygon_translator.bilingual_dict
    detector = AutoPunDetector()

    setup_en, payoff_en = _split_on_contrast(sentence)
    setup_fr = bilingual.translate_text(setup_en).strip()

    if not setup_fr:
        setup_fr = bilingual.translate_text(sentence).strip()

    tail = _pick_tail_noun(pun_word, meaning1, meaning2, sentence, bilingual, detector.stop)
    vibe = _classify_vibe(sentence, meaning1, meaning2)

    payoff_keywords = _compact_keywords_fr(payoff_en or sentence, bilingual, detector.stop)
    compact_tail = tail
    if payoff_keywords:
        priority = [w for w in payoff_keywords if w not in {"voir", "vu", "était", "juste"}]
        if priority:
            compact_tail = priority[-1]

    # Guardrail: if the selected noun is weak or awkward, fall back to the semantically dirtier noun.
    if compact_tail in {"quoi", "coi", "truc"} and tail:
        compact_tail = tail

    noun_phrase = _choose_french_quantifier(compact_tail)

    if vibe == "negative":
        punch = f"mais ce qu’ils ont vu, c’était vraiment dégueu — {noun_phrase}"
    elif vibe == "positive":
        punch = f"et franchement, c’était plutôt malin — presque trop {compact_tail}"
    elif vibe == "absurd":
        punch = f"et là, ça a viré au grand n’importe quoi — du pur {compact_tail}"
    else:
        adjective = "bien trouvé" if candidate else "assez tordu"
        punch = f"et au final, c’était {adjective} — du pur {compact_tail}"

    sentence_fr = f"{setup_fr}… {punch}"
    return _cleanup_french_text(sentence_fr)


## 🚀 Demo

1) Download Lexique 3 from lexique.org (free).
2) Point `LEXIQUE_PATH` to the TSV file.
3) Run the examples below — the notebook now tries to produce a **funny French rendering by default** when it detects a pun.


In [27]:
'''
Summary: example run #1

This cell sets the Lexique file path and runs the full pipeline on example #1
'''


# --- Set this to your local Lexique 3 file path (TSV/CSV) ---
LEXIQUE_PATH = "/content/Lexique383.tsv"  # <-- update this path

translate_pun_complete("The class took a field trip to a meat processing plant, but what they saw was just offal.", lexique_path=LEXIQUE_PATH)



📝 ENGLISH INPUT
   The class took a field trip to a meat processing plant, but what they saw was just offal.
   'what' is OOV → testing portmanteau candidates with context...
   Testing 20 candidates in sentence context...

   📊 Top candidates by context score:
      1. 'wheat' → 1.33
      2. 'hat' → 1.29
      3. 'heat' → 1.12
      4. 'whit' → 1.12
      5. 'wham' → 1.12

   ✅ Selected: 'wheat' + 'hat'

🎯 PUN DETECTED (AUTO)
   Pun word: what
   Meaning A: wheat
   Meaning B: hat

   Details:
Portmanteau: wheat + hat (context-aware)

▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
🔍 POLYGON TRANSLATION ATTEMPTS
▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
   Pun word: 'what'
   Meanings: 'wheat' ↔ 'hat'
   Will try polygons 4 through 8
▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬


🔸 Attempting SQUARE (4-gon)...
   ──────────────────────────────────────────────────────────────────
   Method: EN pun word → FR

'le classe a pris un champ voyage à un viande traitement usine mais coi ils scie était juste déchets'

In [23]:
'''
Summary: example run #2

This cell sets the Lexique file path and runs the full pipeline on example #2
'''

translate_pun_complete("I went to the bank to watch the river bank.", lexique_path=LEXIQUE_PATH)


📝 ENGLISH INPUT
   I went to the bank to watch the river bank.

🎯 PUN DETECTED (AUTO)
   Pun word: bank
   Meaning A: supply
   Meaning B: flight

   Details:
Auto-detected 'bank' as a possible pun word (score 38.00; distance 20; count 2).
Sense A: bank.n.05 — a supply or stock held in reserve for future use (especially in emergencies)
Sense B: bank.n.10 — a flight maneuver; aircraft tips laterally about its longitudinal axis (especially in turning)

▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
🔍 POLYGON TRANSLATION ATTEMPTS
▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
   Pun word: 'bank'
   Meanings: 'supply' ↔ 'flight'
   Will try polygons 4 through 8
▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬


🔸 Attempting SQUARE (4-gon)...
   ──────────────────────────────────────────────────────────────────
   Method: EN pun word → FR synonym/homonym family → Low selection
   English pun word: 'bank'
   Meaning A 'su

'je est allé à le enclose à montre le rivière enclose'

In [24]:
# Diagnostic: is ConceptNet reachable and returning French data?
sem = ConceptNetSemantic(lang="fr")

test_pairs = [
    ("vol",    "voler"),
    ("mouche", "insecte"),
    ("vole",   "mouche"),
    ("voler",  "mouche"),
]

for a, b in test_pairs:
    score = sem.semantic_similarity(a, b)
    related = sem._related_map(a)
    print(f"  sim('{a}', '{b}') = {score:.3f}  |  related terms for '{a}': {list(related.keys())[:5]}")

  sim('vol', 'voler') = 0.000  |  related terms for 'vol': []
  sim('mouche', 'insecte') = 0.000  |  related terms for 'mouche': []
  sim('vole', 'mouche') = 0.000  |  related terms for 'vole': []
  sim('voler', 'mouche') = 0.000  |  related terms for 'voler': []


In [25]:
import pandas as pd

def preview_lexique(path, encoding='utf-8', nrows=5):
    attempts = [
        {"sep": "\t", "engine": "python"},
        {"sep": ";",  "engine": "python"},
        {"sep": ",",  "engine": "python"},
        {"sep": None, "engine": "python"},
    ]
    last_error = None
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, encoding=encoding, on_bad_lines='skip', nrows=nrows, **kwargs)
            if df is not None and len(df.columns) >= 2:
                print(df.columns.tolist())
                print(df.head())
                return df
        except Exception as e:
            last_error = e
    raise ValueError(f"Could not preview Lexique file. Last parser error: {last_error}")

df = preview_lexique("/content/Lexique383.tsv")


['ortho', 'phon', 'lemme', 'cgram', 'genre', 'nombre', 'freqlemfilms2', 'freqlemlivres', 'freqfilms2', 'freqlivres', 'infover', 'nbhomogr', 'nbhomoph', 'islem', 'nblettres', 'nbphons', 'cvcv', 'p_cvcv', 'voisorth', 'voisphon', 'puorth', 'puphon', 'syll', 'nbsyll', 'cv-cv', 'orthrenv', 'phonrenv', 'orthosyll', 'cgramortho', 'deflem', 'defobs', 'old20', 'pld20', 'morphoder', 'nbmorph']
        ortho     phon       lemme cgram genre  nombre  freqlemfilms2  \
0           a        a           a   NOM     m     NaN          81.36   
1           a        a       avoir   AUX   NaN     NaN       18559.22   
2           a        a       avoir   VER   NaN     NaN       13572.40   
3   a capella  akapEla   a capella   ADV   NaN     NaN           0.04   
4  a cappella  akapEla  a cappella   ADV   NaN     NaN           0.04   

   freqlemlivres  freqfilms2  freqlivres  ...    orthrenv  phonrenv  \
0          58.65       81.36       58.65  ...           a         a   
1       12800.81     6350.91    

## 🎯 Interactive Form

Set `LEXIQUE_PATH` once, then try different sentences.

This form now defaults to the notebook's standard behavior: **attempt a joke-first French translation automatically whenever a pun is detected.**


In [26]:
'''
Summary: Colab-style UI form

This cell creates a simple form input so you can paste any sentence and run
the same pipeline interactively with optional detailed logs.

If you want, I can also give you one sentence that explains the “polygon”
idea in plain English (because that’s usually what people ask about first).
'''


# @title 🎭 Enter Your Pun (Scalable) { display-mode: "form" }

LEXIQUE_PATH = "/content/Lexique383.tsv"  # @param {type:"string"}
pun_sentence = "The class took a field trip to a meat processing plant, but what they saw was just offal." # @param {type:"string"}
show_detailed_output = True # @param {type:"boolean"}

result = translate_pun_complete(pun_sentence, lexique_path=LEXIQUE_PATH, show_details=show_detailed_output)



📝 ENGLISH INPUT
   I went to the bank to watch the river bank.

🎯 PUN DETECTED (AUTO)
   Pun word: bank
   Meaning A: supply
   Meaning B: flight

   Details:
Auto-detected 'bank' as a possible pun word (score 38.00; distance 20; count 2).
Sense A: bank.n.05 — a supply or stock held in reserve for future use (especially in emergencies)
Sense B: bank.n.10 — a flight maneuver; aircraft tips laterally about its longitudinal axis (especially in turning)

▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
🔍 POLYGON TRANSLATION ATTEMPTS
▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
   Pun word: 'bank'
   Meanings: 'supply' ↔ 'flight'
   Will try polygons 4 through 8
▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬


🔸 Attempting SQUARE (4-gon)...
   ──────────────────────────────────────────────────────────────────
   Method: EN pun word → FR synonym/homonym family → Low selection
   English pun word: 'bank'
   Meaning A 'su